# NB03 — Reward Analysis, Safety Validation & MLflow Utilities

Analyse the dense reward structure of the Apple Full-Body env, validate
fall detection / safety termination, and define reusable MLflow helpers
for NB04–NB09.

**Apple reward stages:** Reaching → Grasping → Placing → Releasing (max ≈ 10)
**DishWipe reward terms:** 9 terms + 2 balance terms (for NB09 reference)


## Objectives

1. Document Apple reward contract (4 stages + balance penalties).
2. Document DishWipe reward contract (9+2 terms) for NB09 reference.
3. Run test episodes to validate reward range and distribution.
4. Verify `info` dict keys.
5. Validate safety termination (fall detection).
6. Define MLflow helper utilities.
7. Save formal reward contracts as JSON artifacts.


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | Not required | CPU OK |
| RAM | 4 GB | |
| Runtime | ~5-10 min | |


## Imports & Setup


In [ ]:
import sys, os, json, random
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import matplotlib.pyplot as plt

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB03"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify NB01 + NB02 completed
assert (PROJECT_ROOT / "artifacts" / "NB01" / "env_spec_apple.json").exists(), "Run NB01 first!"
assert (PROJECT_ROOT / "artifacts" / "NB02" / "obs_breakdown.json").exists(), "Run NB02 first!"
print("✅ Prerequisites found")


## Configuration


In [ ]:
CFG = {
    "seed": 42,
    "env_id": "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode": "pd_joint_delta_pos",
    "n_test_episodes": 10,
    "n_steps_per_episode": 200,
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("Config loaded")


## Step 1 — Apple Reward Contract

The Apple env uses 4-stage dense reward with balance penalties.
Maximum normalized reward ≈ 10 per step.


In [ ]:
apple_reward_contract = {
    "version": "3.0",
    "env_id": "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "robot": "unitree_g1 (37 DOF, full body, free root)",
    "max_normalized_reward": 10.0,
    "stages": [
        {
            "name": "reaching",
            "description": "Move hand toward apple",
            "formula": "1 - tanh(5 * dist(hand, apple))",
            "max": 1.0,
            "sign": "+",
        },
        {
            "name": "grasping",
            "description": "Close hand around apple",
            "formula": "1.0 if is_grasping(apple)",
            "max": 1.0,
            "sign": "+",
        },
        {
            "name": "placing",
            "description": "Move grasped apple toward bowl",
            "formula": "1 - tanh(5 * dist(apple, bowl))",
            "max": 1.0,
            "sign": "+",
        },
        {
            "name": "releasing",
            "description": "Release apple into bowl",
            "formula": "5.0 if apple_in_bowl AND hand_released",
            "max": 5.0,
            "sign": "+",
        },
    ],
    "balance_terms": [
        {
            "name": "r_balance",
            "description": "Penalty for torso tilt beyond threshold",
            "sign": "-",
        },
        {
            "name": "r_fall",
            "description": "Terminate + large penalty if is_fallen()",
            "sign": "-",
        },
    ],
    "success_condition": "Apple in bowl (dist < 0.05) AND robot standing",
    "failure_conditions": [
        "Robot falls (is_fallen() = True)",
        "Timeout (max_episode_steps exceeded)",
    ],
}

with open(ARTIFACTS_DIR / "reward_contract_apple.json", "w") as f:
    json.dump(apple_reward_contract, f, indent=2)

print("Apple Reward Contract:")
for stage in apple_reward_contract["stages"]:
    print(f"  {stage['sign']} {stage['name']:12s}: max={stage['max']}, {stage['formula']}")
for term in apple_reward_contract["balance_terms"]:
    print(f"  {term['sign']} {term['name']:12s}: {term['description']}")
print(f"\nSuccess: {apple_reward_contract['success_condition']}")


## Step 2 — DishWipe Reward Contract (Reference for NB09)

The DishWipe env uses 9 terms + 2 balance terms. This is documented here
for reference — it will be used in NB09 (bonus task).


In [ ]:
dishwipe_reward_contract = {
    "version": "3.0",
    "env_id": "UnitreeG1DishWipeFullBody-v1",
    "robot": "unitree_g1 (37 DOF, full body, free root)",
    "terms": [
        {"name": "r_clean",   "weight": 10.0,  "sign": "+", "desc": "delta_clean progress"},
        {"name": "r_reach",   "weight": 0.5,   "sign": "+", "desc": "1-tanh(5*dist)"},
        {"name": "r_contact", "weight": 1.0,   "sign": "+", "desc": "is_contacting plate"},
        {"name": "r_sweep",   "weight": 0.3,   "sign": "+", "desc": "lateral movement"},
        {"name": "r_time",    "weight": 0.01,  "sign": "-", "desc": "per step cost"},
        {"name": "r_jerk",    "weight": 0.05,  "sign": "-", "desc": "jerk^2"},
        {"name": "r_act",     "weight": 0.005, "sign": "-", "desc": "action_norm^2"},
        {"name": "r_force",   "weight": 0.01,  "sign": "-", "desc": "excess force"},
        {"name": "r_success", "weight": 50.0,  "sign": "+", "desc": "one-shot at 95%"},
        {"name": "r_balance", "weight": "TBD", "sign": "-", "desc": "penalty for tilt"},
        {"name": "r_fall",    "weight": "TBD", "sign": "-", "desc": "terminate if fallen"},
    ],
    "note": "Used in NB09 only (bonus task)",
}

with open(ARTIFACTS_DIR / "reward_contract_dishwipe.json", "w") as f:
    json.dump(dishwipe_reward_contract, f, indent=2)

print("DishWipe Reward Contract (for NB09):")
for t in dishwipe_reward_contract["terms"]:
    print(f"  {t['sign']} {t['name']:12s}: w={str(t['weight']):>5s}  {t['desc']}")


## Step 3 — Validate Reward with Test Episodes

Run random actions for several episodes. Verify reward is dense
(most steps have non-zero reward) and bounded.


In [ ]:
env = gym.make(
    CFG["env_id"], num_envs=1, obs_mode="state",
    control_mode=CFG["control_mode"], render_mode="rgb_array",
)
env = CPUGymWrapper(env)

all_rewards = []
all_info_keys = set()
termination_counts = {"success": 0, "fall": 0, "timeout": 0}

for ep in range(CFG["n_test_episodes"]):
    obs, info = env.reset(seed=ep * 42)
    ep_rewards = []
    for step in range(CFG["n_steps_per_episode"]):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        ep_rewards.append(float(reward))
        all_info_keys.update(info.keys())

        if terminated:
            if info.get("success", False):
                termination_counts["success"] += 1
            else:
                termination_counts["fall"] += 1
            break
        if truncated:
            termination_counts["timeout"] += 1
            break

    all_rewards.extend(ep_rewards)

reward_stats = {
    "n_episodes":     CFG["n_test_episodes"],
    "total_steps":    len(all_rewards),
    "mean":           float(np.mean(all_rewards)),
    "std":            float(np.std(all_rewards)),
    "min":            float(np.min(all_rewards)),
    "max":            float(np.max(all_rewards)),
    "nonzero_pct":    float(np.mean(np.array(all_rewards) != 0) * 100),
    "positive_pct":   float(np.mean(np.array(all_rewards) > 0) * 100),
    "termination_counts": termination_counts,
}

print("Reward Validation:")
for k, v in reward_stats.items():
    print(f"  {k}: {v}")

with open(ARTIFACTS_DIR / "reward_validation.json", "w") as f:
    json.dump(reward_stats, f, indent=2)


## Step 4 — Info Dict Keys


In [ ]:
info_data = {"available_keys": sorted(list(all_info_keys))}
with open(ARTIFACTS_DIR / "info_keys.json", "w") as f:
    json.dump(info_data, f, indent=2)

print(f"Available info keys ({len(all_info_keys)}):")
for k in sorted(all_info_keys):
    print(f"  - {k}")


## Step 5 — Safety Validation (Fall Detection)

Verify that `is_fallen()` triggers episode termination with random actions.


In [ ]:
try:
    agent = env.unwrapped.agent
    has_is_fallen = hasattr(agent, "is_fallen")
    has_is_standing = hasattr(agent, "is_standing")
except:
    has_is_fallen, has_is_standing = False, False

safety_results = {
    "fall_detection_available":  has_is_fallen,
    "standing_detection_available": has_is_standing,
    "fall_terminates_episode":   termination_counts["fall"] > 0,
    "total_falls":               termination_counts["fall"],
    "total_successes":           termination_counts["success"],
    "total_timeouts":            termination_counts["timeout"],
}

print("Safety Validation:")
for k, v in safety_results.items():
    print(f"  {k}: {v}")

with open(ARTIFACTS_DIR / "safety_validation.json", "w") as f:
    json.dump(safety_results, f, indent=2)


## Step 6 — MLflow Helper Utilities

Define reusable helper functions for NB04–NB09.


In [ ]:
# ── MLflow Helpers (reusable pattern) ──

def setup_mlflow(experiment_name="g1_fullbody_apple_dishwipe"):
    """Set up MLflow tracking. Returns True if successful."""
    try:
        import mlflow
        from dotenv import load_dotenv
        load_dotenv(Path.cwd() / ".env.local")
        uri = os.environ.get("MLFLOW_TRACKING_URI", "")
        if uri:
            mlflow.set_tracking_uri(uri)
            mlflow.set_experiment(experiment_name)
            return True
    except Exception:
        pass
    return False

def log_training_run(run_name, params, metrics=None, artifacts_dir=None):
    """Log a training run to MLflow."""
    try:
        import mlflow
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(params)
            if metrics:
                mlflow.log_metrics(metrics)
            if artifacts_dir:
                mlflow.log_artifacts(str(artifacts_dir))
    except Exception as e:
        print(f"MLflow logging failed: {e}")

def log_eval_run(run_name, eval_dict, artifacts_dir=None):
    """Log an evaluation run to MLflow."""
    try:
        import mlflow
        with mlflow.start_run(run_name=run_name):
            mlflow.log_dict(eval_dict, "eval_results.json")
            if artifacts_dir:
                mlflow.log_artifacts(str(artifacts_dir))
    except Exception as e:
        print(f"MLflow logging failed: {e}")

print("✅ MLflow helper functions defined")
print("  - setup_mlflow(experiment_name)")
print("  - log_training_run(run_name, params, metrics, artifacts_dir)")
print("  - log_eval_run(run_name, eval_dict, artifacts_dir)")


## Plots


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(all_rewards, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].set_title("Reward Distribution (Random Policy)")
axes[0].set_xlabel("Reward")

axes[1].bar(termination_counts.keys(), termination_counts.values(),
            color=["green", "red", "gray"], edgecolor="black")
axes[1].set_title("Termination Reasons")

# Stage visualisation
stages = ["Reaching", "Grasping", "Placing", "Releasing"]
max_rewards = [1.0, 1.0, 1.0, 5.0]
axes[2].bar(stages, max_rewards,
            color=["#FFE082", "#A5D6A7", "#90CAF9", "#CE93D8"],
            edgecolor="black")
axes[2].set_title("Apple Reward Stages (Max per Stage)")
axes[2].set_ylabel("Max Reward")

fig.suptitle("NB03 — Reward & Safety Analysis", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "reward_analysis.png", dpi=150)
plt.show()


## Save Artifacts


In [ ]:
with open(ARTIFACTS_DIR / "nb03_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

env.close()
print("✅ NB03 Reward & Safety Analysis Complete")


## Artifacts

| File | Description |
|------|-------------|
| `reward_contract_apple.json` | Apple 4-stage reward formal contract |
| `reward_contract_dishwipe.json` | DishWipe 9+2 term reward contract (NB09 ref) |
| `reward_validation.json` | Reward range/distribution stats |
| `safety_validation.json` | Fall detection test results |
| `info_keys.json` | Available keys in env info dict |
| `reward_analysis.png` | Reward distribution + termination plots |
